# 金利ボラティリティモデルの学習ノートブック
## Chapter 0-2: 基礎概念とBlack76・Bachelier

このノートブックは、金利ボラティリティモデルを直感と数式、シミュレーション、可視化を通じて学ぶためのものです。

## Chapter 0: Yield Curve Basics

金利モデルを理解する前に、ディスカウント、フォワードレート、スワップレートの基本概念を整理します。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# 日本語フォントの設定（必要に応じて）
plt.rcParams['figure.figsize'] = (12, 6)

# Random seed
np.random.seed(42)

# ===== Basic Functions =====
def discount_factor(zero_rate, T):
    """Discount factor from zero rate"""
    return np.exp(-zero_rate * T)

def zero_rate_from_discount(df, T):
    """Zero rate from discount factor"""
    if T <= 0 or df <= 0:
        return 0
    return -np.log(df) / T

def forward_rate(zero_rates, T1, T2, T_array):
    """Forward rate f(T1, T2) from zero curve"""
    z1 = np.interp(T1, T_array, zero_rates)
    z2 = np.interp(T2, T_array, zero_rates)
    if T2 == T1:
        return z2
    return (z2 * T2 - z1 * T1) / (T2 - T1)

# ===== Yield Curve Scenarios =====
T_array = np.linspace(0.5, 10, 50)

# Flat curve
flat_zero = np.ones_like(T_array) * 0.02

# Upward sloping
upward_zero = 0.02 + 0.01 * (1 - np.exp(-T_array / 5))

# Inverted
inverted_zero = 0.03 - 0.01 * (1 - np.exp(-T_array / 3))

# Compute discount factors
flat_df = discount_factor(flat_zero, T_array)
upward_df = discount_factor(upward_zero, T_array)
inverted_df = discount_factor(inverted_zero, T_array)

# Compute forward rates (instantaneous)
flat_fwd = np.gradient(flat_zero) / np.gradient(T_array) + flat_zero
upward_fwd = np.gradient(upward_zero) / np.gradient(T_array) + upward_zero
inverted_fwd = np.gradient(inverted_zero) / np.gradient(T_array) + inverted_zero

print("Ch0: Yield Curve Basics")
print("=" * 60)
print(f"Flat curve: 0% to {flat_zero[-1]*100:.2f}%")
print(f"Upward curve: {upward_zero[0]*100:.2f}% to {upward_zero[-1]*100:.2f}%")
print(f"Inverted curve: {inverted_zero[0]*100:.2f}% to {inverted_zero[-1]*100:.2f}%")

## Chapter 1: Black 76 Model

Forward rateのオプション価格付けモデル。基本式：
$$dF_t = \sigma_B F_t dW_t$$

特徴：
- Forward rate が正であることを前提
- Lognormal分布
- 相対ボラティリティ (Black volatility)

In [ ]:
# ===== Black 76 Formulas =====
def black76_price(F, K, T, vol, P=1.0):
    """Black 76 call option price"""
    if vol <= 0 or T <= 0 or F <= 0 or K <= 0:
        return max(F - K, 0) * P
    d1 = (np.log(F/K) + 0.5*vol**2*T) / (vol*np.sqrt(T))
    d2 = d1 - vol*np.sqrt(T)
    return P * (F*norm.cdf(d1) - K*norm.cdf(d2))

def black76_delta(F, K, T, vol, P=1.0):
    """Black 76 delta"""
    if vol <= 0 or T <= 0 or F <= 0:
        return 1.0 if F > K else 0.0
    d1 = (np.log(F/K) + 0.5*vol**2*T) / (vol*np.sqrt(T))
    return P * norm.cdf(d1)

def black76_gamma(F, K, T, vol, P=1.0):
    """Black 76 gamma (per 1% move)"""
    if vol <= 0 or T <= 0 or F <= 0:
        return 0.0
    d1 = (np.log(F/K) + 0.5*vol**2*T) / (vol*np.sqrt(T))
    return P * norm.pdf(d1) / (F * vol * np.sqrt(T))

def black76_vega(F, K, T, vol, P=1.0):
    """Black 76 vega (per 1% vol change)"""
    if vol <= 0 or T <= 0 or F <= 0:
        return 0.0
    d1 = (np.log(F/K) + 0.5*vol**2*T) / (vol*np.sqrt(T))
    return P * F * norm.pdf(d1) * np.sqrt(T) * 0.01

# Test
F, K, T, vol = 0.05, 0.05, 1.0, 0.20
price = black76_price(F, K, T, vol)
delta = black76_delta(F, K, T, vol)
gamma = black76_gamma(F, K, T, vol)
vega = black76_vega(F, K, T, vol)

print("Ch1: Black 76 Model")
print("=" * 60)
print(f"Forward: {F*100:.1f}%, Strike: {K*100:.1f}%, T: {T:.1f}y, Vol: {vol*100:.1f}%")
print(f"Price:  {price*10000:.4f} bps")
print(f"Delta:  {delta:.4f}")
print(f"Gamma:  {gamma:.6f} (per 1%)")
print(f"Vega:   {vega:.6f} (per 1% vol)")

## Chapter 2: Bachelier / Normal Model

Forward rateの絶対ボラティリティモデル。基本式：
$$dF_t = \sigma_N dW_t$$

特徴：
- 絶対ボラティリティ (Normal volatility)
- マイナス金利に対応
- 低金利環境で Black vol より安定

In [ ]:
# ===== Bachelier Formulas =====
def bachelier_price(F, K, T, vol_normal, P=1.0):
    """Bachelier caplet price"""
    if vol_normal <= 0 or T <= 0:
        return max(F - K, 0) * P
    d = (F - K) / (vol_normal * np.sqrt(T))
    return P * ((F - K)*norm.cdf(d) + vol_normal*np.sqrt(T)*norm.pdf(d))

def bachelier_delta(F, K, T, vol_normal, P=1.0):
    """Bachelier delta"""
    d = (F - K) / (vol_normal * np.sqrt(T))
    return P * norm.cdf(d)

def bachelier_gamma(F, K, T, vol_normal, P=1.0):
    """Bachelier gamma"""
    d = (F - K) / (vol_normal * np.sqrt(T))
    return P * norm.pdf(d) / (vol_normal * np.sqrt(T))

def bachelier_vega(F, K, T, vol_normal, P=1.0):
    """Bachelier vega (price change per unit vol change)"""
    if vol_normal <= 0 or T <= 0:
        return 0.0
    d = (F - K) / (vol_normal * np.sqrt(T))
    return P * np.sqrt(T) * norm.pdf(d)

# Test
F, K, T, vol_n = 0.02, 0.02, 1.0, 0.005
price_b = bachelier_price(F, K, T, vol_n)
delta_b = bachelier_delta(F, K, T, vol_n)
vega_b = bachelier_vega(F, K, T, vol_n)

print("Ch2: Bachelier / Normal Model")
print("=" * 60)
print(f"Forward: {F*100:.1f}%, Strike: {K*100:.1f}%, T: {T:.1f}y")
print(f"Normal Vol: {vol_n*10000:.0f} bps")
print(f"Price:  {price_b*10000:.4f} bps")
print(f"Delta:  {delta_b:.4f}")
print(f"Vega:   {vega_b:.6f} (per 1bp vol)")